# Numerical data on fenestration and shading variation on ventilation and energy performance of a BESTEST office Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.2320-57pf/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in FAIR^2 are referenced by their `@id`, which uniquely identifies each element. We'll list available record sets and their associated fields for exploration.

In [ ]:
# List all record sets and fields by @id
record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []

if not record_sets:
    print("No record sets found in metadata.")
else:
    for record_set in record_sets:
        print(f"RecordSet @id: {record_set['@id']}")
        # Fetch associated fields with their @id
        fields = record_set.get('field', [])
        if fields:
            for field in fields:
                print(f"  Field @id: {field['@id']}, name: {field.get('name', field['@id'])}")
        else:
            print("  No fields listed for this record set.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.
If no record sets were discovered, you may need to refer to the Croissant JSON and recordSet list, or try fetching using typical EnergyPlus simulation output naming conventions.

In [ ]:
# Attempting to programmatically gather available record sets from the schema
# If recordSets is empty, you may need to manually specify based on FAIR^2 schema or documentation
record_sets = []
if hasattr(metadata, 'recordSet'):
    for rs in metadata.recordSet:
        record_sets.append(rs['@id'])

# For demonstration, let's assume a main record set exists with @id matching EnergyPlus simulation outputs
# Common @id format for Croissant datasets
# Use the first available record set if present
main_record_set_id = None
if record_sets:
    main_record_set_id = record_sets[0]

dataframes = {}

if main_record_set_id is not None:
    # Load records for the main record set
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df
    print(f"Columns in record set '{main_record_set_id}':\n{df.columns.tolist()}\n")
    display(df.head())
else:
    print("No record set IDs found. Please refer to the Croissant schema documentation for available record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Operations may include removing outliers, transforming distributions, or grouping records by meaningful attributes for deeper analysis.
**All entities referenced strictly by their `@id`.**

In [ ]:
# EDA: Example with filtering and normalization
# We'll pick a numeric field by @id, e.g., 'cooling_energy_demand' or similar EnergyPlus output.

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]

    # Try to select a numeric field for demonstration; fallback to first numeric column
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Could be something like '@energyPlusCoolingDemand', depends on actual field @id
        print(f"Using numeric field: {numeric_field_id}")
        
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Group by a categorical or grouping field, if available
        group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"Grouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric fields available in this record set.")
else:
    print("Record set DataFrame not available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
Reference all entities by their `@id` in axis labels or legends.

In [ ]:
# Visualization example: Histogram and scatter plot
if main_record_set_id is not None and numeric_fields:
    df = dataframes[main_record_set_id]
    numeric_field_id = numeric_fields[0]

    plt.figure(figsize=(6, 4))
    plt.hist(df[numeric_field_id], bins=30, color='skyblue', edgecolor='k')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    # Scatter plot if another numeric field exists
    if len(numeric_fields) > 1:
        y_field_id = numeric_fields[1]
        plt.figure(figsize=(6,4))
        plt.scatter(df[numeric_field_id], df[y_field_id], alpha=0.5)
        plt.xlabel(numeric_field_id)
        plt.ylabel(y_field_id)
        plt.title(f'Scatter: {numeric_field_id} vs. {y_field_id}')
        plt.show()
else:
    print("Visualization not possible: missing numeric fields or record set.")

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset using `mlcroissant`, referenced all entities by their `@id`, and explored simulation-based building energy outputs. We performed basic EDA, filtering and normalizing numeric fields, and visualized distributions. This provides a scalable template for Croissant datasets, ensuring reproducibility and traceability in data science workflows.